# Fine-tuning Qwen2.5-VL for Document Intelligence

This notebook shows how to fine-tune Qwen2.5-VL-7B-Instruct on your labeled document dataset  
using **LoRA** via `peft` + `trl` SFTTrainer.

**Requirements:** 24 GB VRAM (A10G, RTX 3090, RTX 4090)

Install extra deps first:
```bash
pip install torch transformers peft trl datasets accelerate bitsandbytes
```

In [ ]:
# ── Imports ──────────────────────────────────────────────────────
import json
from pathlib import Path

import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model
from transformers import (
    AutoProcessor,
    Qwen2VLForConditionalGeneration,
    BitsAndBytesConfig,
)
from trl import SFTConfig, SFTTrainer

MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
DATA_DIR = Path("../data/processed")
OUTPUT_DIR = Path("../models/finetune_output")

In [ ]:
# ── 1. Load labeled dataset ──────────────────────────────────────
# Expected format: JSONL with {"image_path": "...", "output": "{...json...}"}
records = []
jsonl = DATA_DIR / "labels.jsonl"
if jsonl.exists():
    with open(jsonl) as f:
        for line in f:
            records.append(json.loads(line))

print(f"Loaded {len(records)} training examples")

In [ ]:
# ── 2. Format as chat messages ───────────────────────────────────
from PIL import Image
import base64, io

def encode_image(path):
    img = Image.open(path).convert("RGB")
    buf = io.BytesIO()
    img.save(buf, format="JPEG")
    return base64.b64encode(buf.getvalue()).decode()

SYSTEM = "You are a document extraction expert. Respond only with valid JSON."
PROMPT = "Extract all structured data from this document. Return JSON."

def to_chat(record):
    b64 = encode_image(record["image_path"])
    return {
        "messages": [
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
                {"type": "text", "text": PROMPT},
            ]},
            {"role": "assistant", "content": record["output"]},
        ]
    }

dataset = Dataset.from_list([to_chat(r) for r in records])
dataset = dataset.train_test_split(test_size=0.1)
print(dataset)

In [ ]:
# ── 3. Load model in 4-bit ───────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

processor = AutoProcessor.from_pretrained(MODEL_ID)

In [ ]:
# ── 4. Attach LoRA ───────────────────────────────────────────────
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# ── 5. Train ─────────────────────────────────────────────────────
training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=10,
    save_steps=100,
    eval_strategy="steps",
    eval_steps=50,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=processor.tokenizer,
)

trainer.train()

In [ ]:
# ── 6. Save ──────────────────────────────────────────────────────
model.save_pretrained(str(OUTPUT_DIR / "lora_adapter"))
processor.save_pretrained(str(OUTPUT_DIR / "lora_adapter"))
print(f"Saved to {OUTPUT_DIR / 'lora_adapter'}")